# __Preprocesamiento de los datos__

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import holidays
import seaborn as sns

## Cargar datos

In [ ]:
df1 = pd.read_excel('../data/Afluencia_2021.xlsx')
df2 = pd.read_excel('../data/Afluencia_2022.xlsx')
df3 = pd.read_excel('../data/Afluencia_2023.xlsx')
df4 = pd.read_excel('../data/Afluencia_2024.xlsx')
df5 = pd.read_excel('../data/Afluencia_2025.xlsx')

df_total = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)
df_total

In [ ]:
df_total.info()

## Limpiar datos

In [ ]:
df_total.head(5)

In [ ]:
# convertir valores nulos a cero
df_total.fillna(0, inplace=True)
df_total.head(5)

In [ ]:
# convertir campos float a enteros
cols_float = df_total.select_dtypes(include='float').columns
df_total[cols_float] = df_total[cols_float].astype(int)
df_total.info()

In [ ]:
# convertir campo Fecha a datetime
df_total['Fecha'] = pd.to_datetime(df_total['Fecha'], dayfirst=True)
df_total.info()

In [ ]:
# quitar la palabra Linea del campo Linea
df_total['Linea'] = df_total['Linea'].str.replace('LÍNEA', '', regex=False)
df_total['Linea'] = df_total['Linea'].str.strip()
df_total.sample(10)

In [ ]:
# cambiar nombre de la variable 'Total general (Número de pasajeros)'
df_total = df_total.rename(columns={'Total general (Número de pasajeros)':'Total pasajeros'})
df_total.sample()

In [ ]:
# cambiar nombre de las horas
df_total.columns = (
    ['Fecha', 'Linea'] +
    [str(col)[:2] for col in df_total.columns[2:22]] +
    ['Total Pasajeros']
)

df_total.head()

## Features ingening

In [ ]:
# crear un campo que almacene el respectivo año de cada fecha
df_total['Year'] = df_total['Fecha'].dt.year
df_total.head()

In [ ]:
# crear un campo que almacene los dias de la semana
df_total['Day'] = df_total['Fecha'].dt.day_name()
df_total.head()

In [ ]:
# crear un campo que almacene los meses de cada fecha
df_total['Month'] = df_total['Fecha'].dt.month_name()
df_total.sample(5)

## EDA

### Promedio de pasajeros por hora para el periodo que va del 2021 al 2025 

In [ ]:
# listar las horas, lineas y años
horas = [c for c in df_total.columns if c not in ("Fecha", "Linea", "Year", "Total Pasajeros", 'Month', 'Day')]
lineas = sorted(df_total["Linea"].unique())
year = sorted(df_total["Year"].unique())

# transformar formato de los datos en largo
df_long = df_total.melt(
    id_vars=["Fecha", "Linea", "Year"],
    value_vars=horas,
    var_name="Hora",
    value_name="Pasajeros"
)

# obtener el promedio de pasajeros para cada año, por linea y por hora.
promedio = (
    df_long
    .groupby(["Year", "Linea", "Hora"])["Pasajeros"]
    .mean()
    .reset_index()
)

# ordenar los datos cronologicamente
promedio["Hora_idx"] = pd.Categorical(promedio["Hora"], categories=horas, ordered=True)
promedio = promedio.sort_values(["Year", "Linea", "Hora_idx"])
promedio

In [ ]:
# un color para cada año
PALETA = {
    2021: "#378ADD",
    2022: "#1D9E75",
    2023: "#BA7517",
    2024: "#7F77DD",
    2025: "#D85A30",
}
 
horas = [f"{h:02d}" for h in range(4, 24)]
HORA_LABELS = [h[:5] for h in horas]

# obtener cantidad de lineas
n_lineas = len(lineas)

# crear una subgrafica para cada linea del metro
fig, axes = plt.subplots(
    n_lineas, 1,
    figsize=(10, 3.8 * n_lineas),
    sharex=True,
    facecolor="#0e1117"
)

if n_lineas == 1:
    axes = [axes]

# definir color de fondo de la figura 
fig.patch.set_facecolor("#0e1117")
 
for ax, linea in zip(axes, lineas):
    ax.set_facecolor("#161b22")
    ax.spines[:].set_color("#30363d")
    ax.tick_params(colors="#8b949e", labelsize=9)
    ax.yaxis.label.set_color("#c9d1d9")
    ax.xaxis.label.set_color("#c9d1d9")
 
    datos_linea = promedio[promedio["Linea"] == linea].copy()
    y_max_global = 0
 
    for año in year:
        datos_año = datos_linea[datos_linea["Year"] == año].copy()
        if datos_año.empty:
            continue
        y = datos_año["Pasajeros"].values
        x = np.arange(len(horas))
        color = PALETA[año]
        ax.fill_between(x, y, alpha=0.18, color=color)
        ax.plot(x, y, color=color, linewidth=1.8, label=str(año))
        y_max_global = max(y_max_global, y.max())
 
    # Marcar hora pico del promedio total
    promedio_total = datos_linea.groupby("Hora_idx", observed=True)["Pasajeros"].mean().values
    idx_pico = np.argmax(promedio_total)
    ax.axvline(idx_pico, color="#f0f0f0", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.annotate(
        f"Pico: {HORA_LABELS[idx_pico]}",
        xy=(idx_pico, y_max_global * 0.95),
        xytext=(idx_pico + 0.4, y_max_global * 0.95),
        fontsize=8, color="#f0f0f0", alpha=0.7,
        va="top"
    )
 
    ax.set_ylabel("Promedio pasajeros", fontsize=9, color="#8b949e")
    ax.set_title(f"Línea {linea}", fontsize=12, fontweight="bold",
                 color="#e6edf3", loc="left", pad=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: f"{int(v):,}".replace(",", ".")
    ))
    ax.set_ylim(0, y_max_global * 1.12)
    ax.grid(axis="y", color="#30363d", linewidth=0.6, linestyle=":")
    ax.grid(axis="x", color="#30363d", linewidth=0.4, linestyle=":")
 
    # Leyenda dentro del panel
    leg = ax.legend(
        title="Año", title_fontsize=8,
        fontsize=8, loc="upper right",
        framealpha=0.3, frameon=True,
        labelcolor="#c9d1d9",
        facecolor="#161b22", edgecolor="#30363d"
    )
    leg.get_title().set_color("#8b949e")
 
# Eje X compartido
axes[-1].set_xticks(np.arange(len(horas)))
axes[-1].set_xticklabels(HORA_LABELS, rotation=0, ha="center", fontsize=8, color="#8b949e")
axes[-1].set_xlabel("Hora del día", fontsize=10, color="#8b949e")
 
fig.suptitle(
    "Promedio de pasajeros por hora\nMetro de Medellín (2021–2025)",
    fontsize=15, fontweight="bold", color="#e6edf3",
    y=1.005
)

plt.show()

### Heatmap de hora x linea

In [ ]:
# clasificar los dias por habiles y fin de semana
df_total['tipo_dia'] = df_total['Fecha'].dt.day_of_week.map(
    lambda d: 'habil' if d < 5 else 'fin de semana'
)

# clasificar los dias festivos como fin de semana
festivos = holidays.Colombia(years=range(2021, 2026))
df_total.loc[df_total['Fecha'].isin(festivos), 'tipo_dia'] = 'fin de semana'

# convertir de formato ancho a largo
horas = [c for c in df_total.columns if c not in ('Fecha', 'Linea', 'tipo_dia', 'Total Pasajeros', 'Year', 'Day', 'Month')]

df_long = df_total.melt(
    id_vars=['Linea', 'tipo_dia'],
    value_vars=horas,
    var_name='Hora',
    value_name='Pasajeros'
)

df_long.sample(5)

In [ ]:
# funcion de la matriz linea x hora
def build_matrix(tipo):
    subset = df_long[df_long['tipo_dia'] == tipo]
    matriz = subset.pivot_table(
        index='Linea',
        columns='Hora',
        values='Pasajeros',
        aggfunc='mean'
    )

    # ordenar columnas cronologicamente
    matriz = matriz[sorted(matriz.columns)]
    return matriz

# matriz de los dias habiles y fines de semana
mat_habil = build_matrix('habil')
mat_finde = build_matrix('fin de semana')

# definir la misma escala de color a cada heatmap
vmax = max(mat_habil.values.max(), mat_finde.values.max())
paleta = 'YlOrRd'

# crear grafica de heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
hora_labels = [h[:5] for h in mat_habil.columns]

for ax, mat, titulo in [(ax1, mat_habil, "Días hábiles (Lun–Vie)"), (ax2, mat_finde, "Fin de semana (Sáb–Dom)")]:

    sns.heatmap(
        mat, ax=ax,
        cmap=paleta, vmin=0, vmax=vmax,
        xticklabels=hora_labels,
        linewidths=0.4, linecolor="#ffffff20",
        annot=False,           # True si quieres números en celdas
        fmt=".0f", cbar=True
    )

    ax.set_title(titulo, fontsize=13, fontweight="bold", pad=10)
    ax.set_xlabel("Hora del día")
    ax.set_ylabel("Línea")
    ax.tick_params(axis="x", rotation=0)

    fig.suptitle("Flujo promedio de pasajeros — Metro de Medellín",
             fontsize=15, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("heatmap_metro.png", dpi=150, bbox_inches="tight")
plt.show()